In [1]:
import os
import math
import random
import pickle
import json
from dataclasses import dataclass
from collections import Counter, defaultdict, namedtuple
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import LambdaLR

from Bert4Rec_model import BERT4Rec, BERT4RecDataset

print("Python-ready notebook. Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Python-ready notebook. Torch version: 2.5.1+cu121
CUDA available: True


device(type='cuda')

## Đọc dữ liệu

In [2]:
# # Đọc file của Phát
ratings = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\ratings.dat'
movies = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\movies.dat'
users = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\users.dat'

In [3]:
def load_data(ratings_file):
    ratings = pd.read_csv(
        ratings_file,
        sep="::",
        engine="python",
        names=["user", "item", "rating", "timestamp"]
    )

    if not ratings.empty:
        print("Ratings data loaded successfully.")
    else:
        print("Failed to load ratings data.")

    required_cols = ["user", "item", "rating", "timestamp"]
    if all(col in ratings.columns for col in required_cols):
        print("Ratings data contains all required columns.")
    else:
        missing_cols = [col for col in required_cols if col not in ratings.columns]
        print(f"Ratings data is missing columns: {missing_cols}")
    return ratings

# Đọc file của Giang    
# ratings = Path("./data/ratings.dat")
ratings = load_data(ratings)
ratings.head()

Ratings data loaded successfully.
Ratings data contains all required columns.


,user,item,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [4]:
# Sort theo user và timestamp 
ratings = ratings.sort_values(by=['user', 'timestamp']).reset_index(drop=True)

In [5]:
def check_dataframe_info(df, name):
    print(f"=========================================")
    print(f"THÔNG TIN BẢNG: {name.upper()}")
    print(f"=========================================")
    # df.shape trả về một tuple: (số dòng, số cột)
    print(f"1. Số lượng mẫu (Dòng): {df.shape[0]:,}")
    print(f"2. Số lượng đặc trưng (Cột): {df.shape[1]}")
    print(f"3. Kiểu dữ liệu của từng cột:")
    # df.dtypes in ra kiểu dữ liệu của tất cả các cột
    print(df.dtypes)
    print("\n")

# Chạy hàm kiểm tra cho từng DataFrame
check_dataframe_info(ratings, "Ratings")

THÔNG TIN BẢNG: RATINGS
1. Số lượng mẫu (Dòng): 1,000,209
2. Số lượng đặc trưng (Cột): 4
3. Kiểu dữ liệu của từng cột:
user         int64
item         int64
rating       int64
timestamp    int64
dtype: object




## K-Core Filtering

In [6]:
# kiểm tra movieID (item) có liên tục không
print("=== Movie ID (item) ===")
print("Số ID nhỏ nhất: ", ratings["item"].min())
print("Số ID lớn nhất: ", ratings["item"].max())
print("Số lượng ID duy nhất: ", ratings["item"].nunique())

# kiểm tra userID (user) có liên tục không
print("\n=== User ID (user) ===")
print("Số ID nhỏ nhất: ", ratings["user"].min())
print("Số ID lớn nhất: ", ratings["user"].max())
print("Số lượng ID duy nhất: ", ratings["user"].nunique())

=== Movie ID (item) ===
Số ID nhỏ nhất:  1
Số ID lớn nhất:  3952
Số lượng ID duy nhất:  3706

=== User ID (user) ===
Số ID nhỏ nhất:  1
Số ID lớn nhất:  6040
Số lượng ID duy nhất:  6040


Max MovieID là 3952 nhưng thực tế chỉ có 3706 phim  
=> Re-index

In [7]:
# K-core filtering
def k_core_filter(ratings, min_user_interactions, min_item_interactions):
    while True:
        before_size = len(ratings) # lưu số lượng trước khi lọc

        user_counts = ratings["user"].value_counts() # đếm số lượng tương tác của mỗi user
        valid_users = user_counts[user_counts >= min_user_interactions].index # lấy danh sách user có đủ tương tác
        ratings = ratings[ratings["user"].isin(valid_users)] # lọc ratings

        item_counts = ratings["item"].value_counts() # đếm số lượng tương tác của mỗi item
        valid_items = item_counts[item_counts >= min_item_interactions].index # lấy danh sách item có đủ tương tác
        ratings = ratings[ratings["item"].isin(valid_items)] # lọc ratings

        after_size = len(ratings) # lưu số lượng sau khi lọc

        if before_size == after_size:
            break

    return ratings

rating_copy3 = ratings.copy()
rating_copy5 = ratings.copy()

# k = 3
# min_user_interactions = min_item_interactions = k

rating_copy3 = k_core_filter(rating_copy3, 3, 3)
rating_copy5 = k_core_filter(rating_copy5, 5, 5)

print("=== K-core filtering (k=3) ===")
print("Users:", rating_copy3["user"].nunique())
print("Items:", rating_copy3["item"].nunique())
print("Interactions:", len(rating_copy3))
print("Min user interactions:", rating_copy3["user"].value_counts().min())
print("Min item interactions:", rating_copy3["item"].value_counts().min())

print("\n=== K-core filtering (k=5) ===")
print("Users:", rating_copy5["user"].nunique())
print("Items:", rating_copy5["item"].nunique())
print("Interactions:", len(rating_copy5))
print("Min user interactions:", rating_copy5["user"].value_counts().min())
print("Min item interactions:", rating_copy5["item"].value_counts().min())

=== K-core filtering (k=3) ===
Users: 6040
Items: 3503
Interactions: 999917
Min user interactions: 19
Min item interactions: 3

=== K-core filtering (k=5) ===
Users: 6040
Items: 3416
Interactions: 999611
Min user interactions: 18
Min item interactions: 5


In [8]:
# lấy k=3
ratings = rating_copy3 
ratings.head()

,user,item,rating,timestamp
0,1,3186,4,978300019
1,1,1270,5,978300055
2,1,1721,4,978300055
3,1,1022,5,978300055
4,1,2340,3,978300103


## Đánh lại thứ tự bắt đầu từ 1

In [9]:
def reindex_items(ratings):
    unique_items = sorted(ratings["item"].unique())

    movie2id = {
        raw_id: idx + 1
        for idx, raw_id in enumerate(unique_items)
    }

    id2movie = {
        idx + 1: raw_id
        for idx, raw_id in enumerate(unique_items)
    }

    ratings["item"] = ratings["item"].map(movie2id)

    num_items = len(movie2id)

    return ratings, movie2id, id2movie, num_items

ratings, movie2id, id2movie, num_items = reindex_items(ratings)

In [10]:
# # kiểm tra item đã được reindex chưa
# print(ratings["item"].min())
# print(ratings["item"].max())
# print(ratings["item"].nunique())

In [11]:
# tạo sequence
user_sequences = ratings.groupby('user')['item'].apply(list).to_dict()

## Chia dữ liệu (Leave-One-Out)

In [12]:
# leave-one-out split
MAX_LEN    = 200

train_seqs = []   # list of list[int]  — chuỗi con dùng để train
val_seqs   = []   # (input_seq, target_item)
test_seqs  = []   # (input_seq, target_item)

for user, seq in user_sequences.items():

    # Test: item cuối
    test_target = seq[-1] # lấy item cuối
    test_input  = seq[:-1] # tất cả trừ item cuối

    # Val: item áp chót
    val_target  = seq[-2] # lấy item áp chót
    val_input   = seq[:-2] # tất cả trừ 2 item cuối

    # Train: tất cả trừ 2 item cuối  → sliding window
    train_raw = seq[:-2]
    for end in range(2, len(seq[:-2]) + 1):
        sub = train_raw[:end]
        # RIGHT-PADDING với 0  (giống BERT4RecDataset trong file model)
        if len(sub) > MAX_LEN:
            sub = sub[-MAX_LEN:]
        train_seqs.append(sub)

    val_seqs.append((val_input,  val_target))
    test_seqs.append((test_input, test_target))

print(f"Train sequences (sau sliding window) : {len(train_seqs):,}")
print(f"Val  users                           : {len(val_seqs):,}")
print(f"Test users                           : {len(test_seqs):,}")

Train sequences (sau sliding window) : 981,797
Val  users                           : 6,040
Test users                           : 6,040


## Model

### Config

In [ ]:
MAX_LEN = 200
MASK_RATIO = 0.2
BATCH_SIZE = 256
HIDDEN_SIZE = 64
N_LAYERS = 2
N_HEADS = 2
INNER_SIZE = 256
DROPOUT = 0.2
LR = 1e-3
WARMUP_EPOCHS = 10
TOTAL_EPOCHS = 200

# MAX_LEN = 200
# MASK_RATIO = 0.2
# BATCH_SIZE = 256
# HIDDEN_SIZE = 128   
# N_LAYERS = 4
# N_HEADS = 4
# INNER_SIZE = 256
# DROPOUT = 0.2
# LR = 5e-4
# WARMUP_EPOCHS = 10
# TOTAL_EPOCHS = 200

### BERT4REC DATASET & DATALOADER

In [14]:
train_dataset = BERT4RecDataset(
    sequences=train_seqs,
    n_items=num_items,
    max_seq_len=MAX_LEN,
    mask_ratio=MASK_RATIO
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

print("Mask length:", train_dataset.mask_item_length)
print("Num batches:", len(train_loader))

Mask length: 40
Num batches: 3836


### BERT4REC

In [15]:
model = BERT4Rec(
    n_items=num_items + 1,
    max_seq_length=MAX_LEN,
    hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    inner_size=INNER_SIZE,
    hidden_dropout_prob=DROPOUT,
    attn_dropout_prob=DROPOUT,
    hidden_act="gelu",
    mask_ratio=MASK_RATIO,
    loss_type="CE"
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (TOTAL_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("n_items      :", model.n_items)
print("mask_token   :", model.mask_token)
print("embedding:", model.item_embedding.weight.shape)
print("output_bias:", model.output_bias.shape)
print(f"Trainable Params: {n_params:,}")

n_items      : 3504
mask_token   : 3504
embedding: torch.Size([3505, 128])
output_bias: torch.Size([3504])
Trainable Params: 1,024,688


In [16]:
def build_test_tensors(seqs, mask_token, max_len):
    X, Y = [], []
    for input_seq, target in seqs:
        seq = input_seq + [mask_token]
        seq = seq[-max_len:]
        seq = [0]*(max_len-len(seq)) + seq
        X.append(seq)
        Y.append(target)
    return (torch.LongTensor(X), torch.LongTensor(Y))

val_X, val_Y = build_test_tensors(val_seqs, model.mask_token, MAX_LEN)
test_X, test_Y = build_test_tensors(test_seqs, model.mask_token, MAX_LEN)

val_loader = DataLoader(TensorDataset(val_X, val_Y), batch_size=256, shuffle=False)
test_loader = DataLoader(TensorDataset(test_X, test_Y), batch_size=256, shuffle=False)

print(f"Val  users : {len(val_Y):,}")
print(f"Test users : {len(test_Y):,}")

Val  users : 6,040
Test users : 6,040


In [17]:
# ── Tính popularity distribution từ train_seqs ──────────────
# Chạy 1 lần trước khi gọi evaluate
all_items_flat   = [item for seq in train_seqs for item in seq]
item_counts      = Counter(all_items_flat)
pop_items        = np.array(list(item_counts.keys()),   dtype=np.int64)
pop_counts       = np.array(list(item_counts.values()), dtype=np.float32)
popularity_probs = pop_counts / pop_counts.sum()

@torch.no_grad()
def evaluate(model, loader, device, ks=(1, 5, 10), num_neg=99):
    model.eval()
    metrics = {f"HR@{k}": 0.0 for k in ks}
    metrics.update({f"NDCG@{k}": 0.0 for k in ks})
    metrics["MRR"] = 0.0
    total = 0

    for masked_seq, targets in loader:
        masked_seq = masked_seq.to(device)
        targets    = targets.to(device)

        seq_out    = model.forward(masked_seq)
        last_out   = seq_out[:, -1, :]                             # [B, H]
        item_emb   = model.item_embedding.weight[: model.n_items]  # [n_items, H]
        all_logits = torch.matmul(last_out, item_emb.transpose(0, 1)) \
                     + model.output_bias                            # [B, n_items]

        for i in range(targets.size(0)):
            target = targets[i].item()

            # Sample 99 negatives theo popularity distribution (đúng paper gốc)
            negs = []
            while len(negs) < num_neg:
                neg = int(np.random.choice(pop_items, p=popularity_probs))
                if neg != target and neg not in negs:
                    negs.append(neg)

            # 100 candidates: index 0 = target, index 1..99 = negatives
            candidates  = [target] + negs
            cand_tensor = torch.tensor(candidates, dtype=torch.long, device=device)
            scores      = all_logits[i][cand_tensor]               # [100]

            # Rank của target (index 0) trong 100 candidates
            _, indices = torch.sort(scores, descending=True)
            rank = (indices == 0).nonzero(as_tuple=True)[0].item() + 1

            metrics["MRR"] += 1.0 / rank
            for k in ks:
                if rank <= k:
                    metrics[f"HR@{k}"]   += 1.0
                    metrics[f"NDCG@{k}"] += 1.0 / math.log2(rank + 1)
            total += 1

    if total > 0:
        for key in metrics:
            metrics[key] /= total

    model.train()
    return metrics, total

In [ ]:
EPOCHS          = 100
MODEL_SAVE_PATH = "saved_models/bert4rec_best.pth"
PATIENCE        = 5    # 10 lần eval không cải thiện = dừng
EVAL_EVERY      = 10

### Wandb

In [19]:
# pip install wandb

In [21]:
import wandb

# KHÔNG PUBLIC KEY NÀY
wandb.login(
    key="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"
)

wandb.init(
    project="bert4rec-movielens-1m",
    name="bert4rec_run_01",

    config={
        "epochs": EPOCHS,
        "batch_size": train_loader.batch_size,
        "lr": optimizer.param_groups[0]["lr"],
        "max_len": model.max_seq_length,
        "hidden_size": model.hidden_size,
        "eval_every": EVAL_EVERY,
        "patience": PATIENCE
    }
)

wandb.watch(
    model,
    log="gradients",
    log_freq=100
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\TanPhat\_netrc
wandb: Currently logged in as: lamgiang (lamgiang-fpt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Training

In [24]:
patience_counter = 0
best_val_hr10    = 0.0
best_loss        = float("inf")
history_loss     = []
history_acc      = []
global_step = 0

os.makedirs("saved_models", exist_ok=True)

print("-" * 60) 
print(f"BẮT ĐẦU HUẤN LUYỆN — {EPOCHS} epochs  |  device: {device}")
print(f"Early Stopping: PATIENCE = {PATIENCE} lần eval (mỗi {EVAL_EVERY} epoch)")
print("-" * 60)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0; total_correct = 0; total_masked = 0

    pbar = tqdm(total=len(train_loader),
                desc=f"Epoch {epoch+1:03d}/{EPOCHS}",
                leave=True, dynamic_ncols=True)

    for batch in train_loader:
        masked_item_seq = batch["masked_item_seq"].to(device)
        pos_items       = batch["pos_items"].to(device)
        neg_items       = batch["neg_items"].to(device)
        masked_index    = batch["masked_index"].to(device)

        optimizer.zero_grad()
        loss = model.calculate_loss(masked_item_seq, pos_items, neg_items, masked_index)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)

        global_step += 1

        if global_step % 50 == 0:
            wandb.log({
                "batch_loss": loss.item(),
                "grad_norm": grad_norm.item()
            })

        optimizer.step()
        total_loss += loss.item()

        with torch.no_grad():
            seq_out  = model.forward(masked_item_seq)
            pred_map = model.multi_hot_embed(
                masked_index, masked_item_seq.size(-1)
            ).view(masked_index.size(0), masked_index.size(1), -1)
            gathered = torch.bmm(pred_map, seq_out)
            emb      = model.item_embedding.weight[: model.n_items]
            logits   = torch.matmul(gathered, emb.transpose(0, 1)) + model.output_bias
            preds    = logits.argmax(dim=-1)
            valid    = (masked_index > 0)
            total_correct += ((preds == pos_items) & valid).sum().item()
            total_masked  += valid.sum().item()

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        pbar.update(1)

    scheduler.step()
    print(f"LR hiện tại: {scheduler.get_last_lr()[0]:.6f}")

    avg_loss = total_loss / len(train_loader)
    avg_acc  = total_correct / total_masked if total_masked > 0 else 0.0

    current_lr = scheduler.get_last_lr()[0]

    wandb.log({"epoch": epoch + 1,
                "train_loss": avg_loss,
                "train_acc": avg_acc,
                "learning_rate": current_lr,
                "best_loss_so_far": best_loss})

    history_loss.append(avg_loss)
    history_acc.append(avg_acc * 100)

    if avg_loss < best_loss:
        best_loss = avg_loss   # chỉ để theo dõi, không save ở đây

    pbar.set_postfix_str(f"Loss: {avg_loss:.4f} | Acc: {avg_acc*100:.2f}%")
    pbar.close()

    # ── Validation mỗi EVAL_EVERY epoch ──────────────────────
    if (epoch + 1) % EVAL_EVERY == 0:
        val_metrics, _ = evaluate(model, val_loader, device, ks=(10,))
        val_hr10 = val_metrics["HR@10"]
        wandb.log({
            "epoch": epoch + 1,
            "val_hr10": val_hr10,
            "best_val_hr10": max(
                best_val_hr10,
                val_hr10
            )
        })
        print(f"  → [Val epoch {epoch+1}] HR@10: {val_hr10:.4f}  (best: {best_val_hr10:.4f})")

        if val_hr10 > best_val_hr10:
            best_val_hr10    = val_hr10
            patience_counter = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH)

            wandb.save(MODEL_SAVE_PATH)
            wandb.log({"best_val_hr10": best_val_hr10})

            print(f"  → Saved! Val HR@10 = {val_hr10:.4f}")
        else:
            patience_counter += 1
            print(f"  → Không cải thiện [{patience_counter}/{PATIENCE}]")
            if patience_counter >= PATIENCE:
                print("!" * 60)
                print(f"EARLY STOPPING tại epoch {epoch+1} | Best Val HR@10: {best_val_hr10:.4f}")
                print("!" * 60)
                break
    # ─────────────────────────────────────────────────────────

wandb.summary["best_loss"] = best_loss
wandb.summary["best_val_hr10"] = best_val_hr10

wandb.finish()

print("-" * 60)
print(f"HOÀN THÀNH! Best Loss: {best_loss:.4f} | Best Val HR@10: {best_val_hr10:.4f}")
print(f"Model: {MODEL_SAVE_PATH} | Epochs: {epoch+1}")
print("-" * 60)

------------------------------------------------------------
BẮT ĐẦU HUẤN LUYỆN — 400 epochs  |  device: cuda
Early Stopping: PATIENCE = 5 lần eval (mỗi 10 epoch)
------------------------------------------------------------


Epoch 001/400:   0%|          | 0/3836 [00:00<?, ?it/s]

LR hiện tại: 0.000100


Epoch 002/400:   0%|          | 0/3836 [00:00<?, ?it/s]

LR hiện tại: 0.000150


...

LR hiện tại: 0.000233


Epoch 110/400:   0%|          | 0/3836 [00:00<?, ?it/s]

LR hiện tại: 0.000229
  → [Val epoch 110] HR@10: 0.6858  (best: 0.6927)
  → Không cải thiện [5/5]
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
EARLY STOPPING tại epoch 110 | Best Val HR@10: 0.6927
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


batch_loss,█▆▅▄▃▃▂▃▃▂▂▂▂▂▂▁▁▂▂▂▂▂▁▂▁▂▁▂▂▁▁▁▂▂▁▁▁▁▁▂
best_loss_so_far,█▆▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_hr10,▁▁▄▄███████████
epoch,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
grad_norm,▁▇██▆▆▅▄▅▄▅▅▅▅▅▅▅▅▅▅▅▆▅▅▆▆▆▅▆▅▇▇▆█▆▇▆▇▆▇
learning_rate,▁▄▅▆███████████▇▇▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▄▄▄▃▃
train_acc,▁▁▂▄▄▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇███████████████
train_loss,█▇▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_hr10,▁▄█▅▇█▆▅▆▇▇
batch_loss,4.67289
best_loss,4.76013


------------------------------------------------------------
HOÀN THÀNH! Best Loss: 4.7601 | Best Val HR@10: 0.6927
Model: saved_models/bert4rec_best.pth | Epochs: 110
------------------------------------------------------------


### Evaluation

In [25]:
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(
        torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=True)
    )
    print(f"✅ Loaded: {MODEL_SAVE_PATH}")

print("\n" + "=" * 50)
print("🎯 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST")
print("=" * 50)

test_metrics, n_test = evaluate(model, test_loader, device,
                                ks=(1, 5, 10), num_neg=99)  # ← thêm num_neg=99
for k, v in test_metrics.items():
    print(f"  {k:<10}: {v:.4f}")
print(f"  (Tổng {n_test:,} users)")
print("-" * 50)

print("\n📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP VAL")
print("-" * 50)
val_metrics, n_val = evaluate(model, val_loader, device,
                               ks=(1, 5, 10), num_neg=99)  # ← thêm num_neg=99
for k, v in val_metrics.items():
    print(f"  {k:<10}: {v:.4f}")
print(f"  (Tổng {n_val:,} users)")
print("-" * 50)

✅ Loaded: saved_models/bert4rec_best.pth

🎯 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST
  HR@1      : 0.2639
  HR@5      : 0.5460
  HR@10     : 0.6637
  NDCG@1    : 0.2639
  NDCG@5    : 0.4129
  NDCG@10   : 0.4511
  MRR       : 0.3973
  (Tổng 6,040 users)
--------------------------------------------------

📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP VAL
--------------------------------------------------
  HR@1      : 0.2674
  HR@5      : 0.5623
  HR@10     : 0.6874
  NDCG@1    : 0.2674
  NDCG@5    : 0.4229
  NDCG@10   : 0.4637
  MRR       : 0.4063
  (Tổng 6,040 users)
--------------------------------------------------
